In [ ]:
# Initialize Spark session for data setup

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("FirstStepNotebook") \
    .getOrCreate()

## Create Namespace

In [ ]:
# Create the taxi namespace in Polaris catalog

spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.taxi")

## Download NYC Taxi Data

Download NYC Yellow Taxi trip data for June through October 2023 and upload to MinIO.

In [ ]:
# Download NYC taxi data and upload to MinIO

import urllib.request
import os
import boto3
from botocore.client import Config

# Configure MinIO client
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download and upload NYC taxi data
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
minio_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

os.makedirs(temp_dir, exist_ok=True)

months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    s3_key = f"{minio_prefix}/{filename}"
    
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, local_path)
    
    print(f"Uploading to MinIO: {s3_key}")
    s3_client.upload_file(local_path, bucket_name, s3_key)
    uploaded_files.append(s3_key)
    
    os.remove(local_path)

print(f"\nUploaded {len(uploaded_files)} files to s3://{bucket_name}/{minio_prefix}/")

## Copy Data for Snapshot Example

Copy the raw data to a dedicated location for the snapshot example. This ensures data files are in the correct location when the snapshot table is created.

In [ ]:
# Copy raw data to taxi/trips_snapshot/data/ for snapshot example

source_prefix = "raw/"
dest_prefix = "taxi/trips_snapshot/data/"

# List all files in the raw directory
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=source_prefix)

if 'Contents' in response:
    for obj in response['Contents']:
        source_key = obj['Key']
        # Skip if it's just the directory marker
        if source_key.endswith('/'):
            continue
        
        # Get just the filename
        filename = os.path.basename(source_key)
        dest_key = f"{dest_prefix}{filename}"
        
        print(f"Copying {source_key} -> {dest_key}")
        s3_client.copy_object(
            Bucket=bucket_name,
            CopySource={'Bucket': bucket_name, 'Key': source_key},
            Key=dest_key
        )
    
    print(f"\nCopied data files to s3://{bucket_name}/{dest_prefix}")
else:
    print("No files found in raw directory")

## Verify Data

Verify both data locations are accessible.

In [ ]:
# Verify raw data location

raw_df = spark.read.parquet("s3a://warehouse/raw/")
print(f"Raw data: {raw_df.count():,} rows")

# Verify taxi/trips_snapshot data location
trips_snapshot_df = spark.read.parquet("s3a://warehouse/taxi/trips_snapshot/data/")
print(f"Trips snapshot data: {trips_snapshot_df.count():,} rows")